## First Step | Connect to a local mysql connection

Make a database if not already
Get list of Ticker, download the data

In [ ]:
import sql_init
import stock_init

In [ ]:
file_path = "sql_login.json"
DB_NAME = "asx_stock_data"
connection, mycursor = sql_init.connect_to_mysql(file_path, DB_NAME)

if connection is None:
    print("Failed to connect to MySQL database.")
print("Connected")

In [ ]:
# Get the list of ticker symbols from db
query = """
SELECT *
FROM ticker_list
"""
mycursor.execute(query)
ticker_list = [i[0] for i in mycursor.fetchall()]

In [ ]:
ticker_list[:5]

## Download data
Put download to **True** if we need to download more data

else **False**, this part only needed to be run every now and then

For the next downloads, specify the start date depending on the last date data we have

In [ ]:
download = True
if download == True:
    for ticker in ticker_list:
        data = stock_init.download_ticker(ticker+".AX")
        sql_init.insert_stock_data(mycursor, ticker, data)
        connection.commit()

## Applying Financial Mathematical knowledge
Calculate returns

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Get the list of ticker symbols from db
query = """
SELECT *
FROM stock_data
"""
mycursor.execute(query)
columns = ["symbol", "date", "close", "high", "low", "volume"]

df = pd.DataFrame(mycursor.fetchall(), columns=columns)

#### Verify all the data is as expected

In [ ]:
# Same count of data
counts = df.groupby("symbol")["close"].count()
print(counts.min(), counts.max())  # if min == max, all equal

In [ ]:
df["log_return"] = df.groupby("symbol")["close"].transform(lambda x: np.log(x/x.shift(1)))

In [ ]:
df.dtypes

In [ ]:
df['date'] = pd.to_datetime(df['date'])

In [ ]:
df.head()

#### Create trailing returns
Calculate the n-month trailing returns for every stock, and rank them

Find the top $20$% for every beginging of the month, this will show us which stocks to hold for any given month.
Test the strategy later.

In [ ]:
# 12 month trailing momentum
df = stock_init.trailing_return(df, month=12)

In [ ]:
df['rank'] = df.groupby('date')['momentum'].rank(pct=True)

In [ ]:
first_days = df.groupby(df['date'].dt.to_period('M'))['date'].min()

In [ ]:
monthly_first = df[df['date'].isin(first_days)]
monthly_first[monthly_first["rank"]>=0.8]

#### Rolling Z-score
Find how far a stock is from it's mean of 20 days window

In [ ]:
def rolling_z(window_size = 20):
    rolling_mean = df["close"].rolling(window_size).mean()
    rolling_std = df["close"].rolling(window_size).std()
    return (df["close"]-rolling_mean)/rolling_std

df["z_momentum"] = rolling_z(20)
df["z_reversion"] = rolling_z(252)

In [ ]:
df.tail()

#### Understand the difference between Momentum and Z-Momentum

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,6))

ax.scatter(df["z_momentum"], df["momentum"], alpha=0.5)
ax.set_title("percentage Momentum vs z Momentum")

single_corr = df["z_momentum"].corr(df["momentum"])

ax.set_title(f"Momentum vs Z-Momentum, r = {single_corr:.4f}")
ax.set_ylabel("Momentum")
ax.set_xlabel("Z-Momentum")
plt.show()

Looking at the r score, and the graph, we can conclude that there ay be something different that they capture

- High raw momentum, low z-momentum — a stock that's just inherently volatile. A 30% move might be totally normal for it, so relative to its own history it isn't "unusual," even though the raw number looks big.
- Low raw momentum, high z-momentum — a normally very stable, low-volatility stock that moved 8%. Small in absolute terms, but huge relative to what that stock usually does.
- Regime changes — if a stock's volatility recently spiked or dropped, its rolling std shifts, which changes the z-score even if the raw return pattern looks similar to before

#### Find average dollar volume
This is mainly to filter out stocks that are not liquid, which although due to the selection of stock is trivial, as most asx 50 if not all is highly liquid.

In [ ]:
df["dollar_vol"] = df["close"]*df["volume"]
df["avg_dollar_vol"] = (df.groupby("symbol")["dollar_vol"]
                          .transform(lambda x: x.rolling(20).mean()))

#### Get 50, 200 MA
We will use these to apply the golden cross indicator

In [ ]:
df["ma_50"] = (df.groupby("symbol")["close"]
                          .transform(lambda x: x.rolling(50).mean()))
df["ma_200"] = (df.groupby("symbol")["close"]
                          .transform(lambda x: x.rolling(200).mean()))

#### Close location within the range — conviction signal
- Close near the high (close to 1) → buyers won the day, bullish conviction
- Close near the low (close to 0) → sellers won the day, bearish conviction

In [ ]:
df['close_location'] = (df['close'] - df['low']) / (df['high'] - df['low'])

In [ ]:
df.columns

#### Export signals back to SQL into a new table

In [ ]:
if not sql_init.check_table_exists(mycursor, "signals"):
    query = """
        CREATE TABLE signals(
        symbol              VARCHAR(4) NOT NULL,
        date                DATE NOT NULL,
        log_return          DOUBLE,
        momentum            DOUBLE,
        mom_rank            DOUBLE,
        z_momentum          DOUBLE,
        z_reversion         DOUBLE,
        avg_dollar_vol      DOUBLE,
        ma_50               DOUBLE,
        ma_200              DOUBLE,
        close_location      DOUBLE,
        PRIMARY KEY (symbol, date),
        FOREIGN KEY (symbol, date) REFERENCES stock_data(symbol, date)
        )

    """
    mycursor.execute(query)

In [ ]:
df = df.replace({np.nan: None})
rows = [
    (
        row.symbol,
        row.date,
        row.log_return,
        row.momentum,
        row.rank,
        row.z_momentum,
        row.z_reversion,
        row.avg_dollar_vol,
        row.ma_50,
        row.ma_200,
        row.close_location
    )
    for row in df.itertuples(index=False)
]
query = """
    INSERT INTO signals
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

# Run if needed to insert signals
#mycursor.executemany(query, rows)
#connection.commit()

In [ ]:
sql_init.close_connection(connection)